# 01 Probe set
Run 10 probe tasks 5 times on two models and compare.

Step 1: set up the kit

In [ ]:
# Setup: in Colab, clone the kit first. Locally, find the kit folder. Change REPO_URL to your fork.
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/GITHUB_OWNER/atp-builder-kit"
if "google.colab" in sys.modules and not Path("kit/llm.py").exists():
    if not Path("atp-builder-kit").exists():
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir("atp-builder-kit")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "google-genai==3.3.0", "openai==3.51.0"], check=True)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "kit" / "llm.py").exists())
sys.path.insert(0, str(ROOT))
DATA = Path(os.environ.get("KIT_DATA_DIR", ROOT / "data"))
OUTPUT = Path(os.environ.get("KIT_OUTPUT_DIR", ROOT / "stage-01"))
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Kit folder:", ROOT)

Step 2: pick two models

In [ ]:
from kit.llm import call_llm, offline
from kit.prices import cost
import json, re
import pandas as pd

MODEL_A = os.environ.get("PROBE_MODEL_A", "gemini-2.5-flash")
MODEL_B = os.environ.get("PROBE_MODEL_B", "gemini-2.5-flash-lite")
if offline():
    MODEL_A, MODEL_B = "mock-model-a", "mock-model-b"
RUNS = 5
print(MODEL_A, "vs", MODEL_B)

Step 3: load the probes

In [ ]:
probes = pd.read_csv(DATA / "probes.csv").fillna("")
probes

Step 4: define the pass checks

In [ ]:
def passes(text, check_type, check_value):
    if check_type == "must_contain":
        return check_value in text
    if check_type == "regex":
        return re.search(check_value, text.strip(), re.S) is not None
    if check_type == "valid_json":
        try:
            json.loads(text.strip())
            return True
        except ValueError:
            return False
    raise ValueError(f"Unknown check type {check_type}")

print(passes('{"ok": true}', "valid_json", ""), passes("hi", "must_contain", "bye"))

Step 5: run every probe 5 times on each model

In [ ]:
rows = []
for model in [MODEL_A, MODEL_B]:
    for p in probes.itertuples():
        for run in range(RUNS):
            r = call_llm(p.task, model=model, temperature=0.7, max_output_tokens=300)
            rows.append({"model": model, "probe_id": p.probe_id, "run": run,
                         "passed": passes(r["text"], p.check_type, p.check_value),
                         "cost_usd": cost(r["input_tokens"], r["output_tokens"], model),
                         "latency_s": r["latency_s"]})
runs = pd.DataFrame(rows)
print(len(runs), "calls")

Step 6: pass rate per probe

In [ ]:
runs.pivot_table(index="probe_id", columns="model", values="passed", aggfunc="mean")

Step 7: compare the two models

In [ ]:
summary = runs.groupby("model").agg(pass_rate=("passed", "mean"), cost_per_task=("cost_usd", "mean"),
                                    median_latency_s=("latency_s", "median"))
summary["cost_per_task"] = summary["cost_per_task"].round(6)
summary